# Module 1: Pothole and crack detector (RDD2022) — Kaggle training

Before running:
1. **Settings (right sidebar) → Accelerator → GPU T4 x2** (or P100)
2. **Settings → Internet → On** (needed to clone the repo and install packages; requires a phone-verified account)
3. **Add Input → Datasets → search `rdd-2022` by aliabdelmenam → Add**

For the long run, use **Save Version → Save & Run All (Commit)**. It keeps running with the browser closed, and everything in `/kaggle/working` is kept as the version's output.

Dataset: RDD2022, Arya et al., CC BY-SA 4.0 — attribution required in the report.


## 0. Settings

In [ ]:
MODEL = "yolo11n.pt"      # or "yolov8n.pt"; check against the current Ultralytics docs
IMGSZ = 640
EPOCHS = 50
TIME_HOURS = 5            # hard cap on training time; note Ultralytics then plans epochs to fill it
BATCH = 32                # lower to 16 if you hit GPU out-of-memory
FRACTION = 1.0            # 0.25 for a quick first baseline
RUN_NAME = "rdd_pothole_v1"
WORK = "/kaggle/working"
YOLO_DIR = "/kaggle/working/rdd_yolo"
REPO = "https://github.com/divyanshkulshrestha23/road-anomaly-detection"
# The class-id mapping lives in the repo: detector/configs/rdd_source.yaml (pulled in step 2).


## 1. GPU check and find the attached dataset

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import glob, os
cands = [p for p in glob.glob("/kaggle/input/**/", recursive=True) if "rdd" in p.lower()]
cands.sort(key=len)
assert cands, "RDD2022 not attached: use Add Input -> Datasets -> rdd-2022"
RAW_DIR = cands[0].rstrip("/")
print("Dataset at", RAW_DIR)
print(os.listdir(RAW_DIR)[:20])


## 2. Clone repo and install

In [ ]:
import os
if not os.path.exists(f"{WORK}/road-anomaly-detection"):
    !git clone -q {REPO} {WORK}/road-anomaly-detection
else:
    !git -C {WORK}/road-anomaly-detection pull -q
%cd {WORK}/road-anomaly-detection
!pip install -q -r detector/requirements.txt
!git log --oneline -1


## 3. Inspect the dataset layout
Check: India should be listed, and class names should look like D00/D10/D20/D40 or spelled-out names.

In [ ]:
!python detector/rdd_prepare.py --src {RAW_DIR} --inspect

## 3b. Preview each class id
Writes one image grid per class id (red boxes) to `/kaggle/working/previews/`. Open them from the file browser on the right, or download them, and send them to Claude to confirm which id is which class.

In [ ]:
!python detector/rdd_prepare.py --src {RAW_DIR} --preview {WORK}/previews
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(f"{WORK}/previews/*.jpg")):
    print(f); display(Image(filename=f, width=900))


## 4. Convert to YOLO format (images are symlinked, so this uses little disk)

In [ ]:
# Stops with an error until detector/configs/rdd_source.yaml has the confirmed class names.
!python detector/rdd_prepare.py --src {RAW_DIR} --out {YOLO_DIR} || (echo "STOP: class mapping not set" && exit 1)
import os
assert os.path.exists(f"{YOLO_DIR}/data.yaml"), "Data prep failed; see the message above"


## 5. Train

In [ ]:
from ultralytics import YOLO
project = f"{WORK}/runs"
model = YOLO(MODEL)
model.train(data=f"{YOLO_DIR}/data.yaml", epochs=EPOCHS, time=TIME_HOURS, imgsz=IMGSZ,
            batch=BATCH, fraction=FRACTION, project=project, name=RUN_NAME, exist_ok=True,
            patience=15, seed=42, device=0, workers=4)


## 6. Evaluate: all countries, then India only

In [ ]:
from ultralytics import YOLO
best = f"{WORK}/runs/{RUN_NAME}/weights/best.pt"
model = YOLO(best)
results = {}
for tag, data in [("all", f"{YOLO_DIR}/data.yaml"), ("india", f"{YOLO_DIR}/data_india.yaml")]:
    for split in ["val", "test"]:
        m = model.val(data=data, split=split, imgsz=IMGSZ, batch=BATCH, plots=False, verbose=False)
        per_class = {model.names[i]: round(float(m.box.maps[i]), 4) for i in range(len(model.names))}
        results[f"{tag}_{split}"] = {"mAP50": round(float(m.box.map50), 4),
                                     "mAP50_95": round(float(m.box.map), 4),
                                     "precision": round(float(m.box.mp), 4),
                                     "recall": round(float(m.box.mr), 4),
                                     "per_class_mAP50_95": per_class}
        print(tag, split, results[f"{tag}_{split}"])


## 7. Save summary now (so a later failure cannot lose the results)

In [ ]:
import json, datetime, subprocess, shutil, torch
summary = {
    "run": RUN_NAME, "model": MODEL, "imgsz": IMGSZ, "epochs": EPOCHS, "time_cap_h": TIME_HOURS,
    "batch": BATCH, "fraction": FRACTION, "platform": "kaggle",
    "date": datetime.datetime.now().isoformat(timespec="seconds"),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "repo_commit": subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip(),
    "dataset": "RDD2022 (Kaggle: aliabdelmenam/rdd-2022), CC BY-SA 4.0",
    "targets": {"pothole_mAP50": 0.5},
    "results": results,
}
with open(f"{WORK}/summary_{RUN_NAME}.json", "w") as f:
    json.dump(summary, f, indent=2)
for fn in ["prepare_summary.json", "split_manifest.csv"]:
    shutil.copy2(f"{YOLO_DIR}/{fn}", WORK)
print(json.dumps(summary, indent=2))


## 8. Export float TFLite for the app (INT8 skipped: its calibration ran out of memory)

In [ ]:
!python detector/export_tflite.py --weights {best} --out {WORK}/exports
import shutil
shutil.rmtree(YOLO_DIR, ignore_errors=True)
shutil.rmtree(f"{WORK}/road-anomaly-detection", ignore_errors=True)
